# Distribution Spike Stimulus Example

This notebook demonstrates how to compose distribution blocks and a `DistributionSpikeStimulus`.

It shows how to:

1. Define timing distributions as reusable blocks
2. Reference those distributions from spike stimulus blocks
3. Prepare the resulting blocks for use in a simulation form/config

The examples below use:

- `FloatConstantDistribution` for regular spike trains
- `ExponentialDistribution` for Poisson-like spike trains
- `GammaDistribution` for controllable spike-timing regularity

In [1]:
import obi_one as obi

## Define distribution blocks

Each distribution block defines how inter-spike intervals are sampled.

In [2]:
distributions = {
    "regular_dist": obi.FloatConstantDistribution(value=10.0),  # 10 ms ISI
    "poisson_dist": obi.ExponentialDistribution(scale=100.0, random_seed=42),  # ~10 Hz
    "gamma_dist": obi.GammaDistribution(shape=5.0, scale=20.0, random_seed=42),
}

distributions

{'regular_dist': FloatConstantDistribution(type='FloatConstantDistribution', value=10.0),
 'poisson_dist': ExponentialDistribution(type='ExponentialDistribution', scale=100.0, random_seed=42),
 'gamma_dist': GammaDistribution(type='GammaDistribution', shape=5.0, scale=20.0, random_seed=42)}

## Define neuron set blocks

These neuron sets can be referenced by the spike stimulus blocks.

In [3]:
neuron_sets = {
    "all": obi.AllNeurons(),
}

neuron_sets

{'all': AllNeurons(type='AllNeurons', sample_percentage=100.0, sample_seed=1)}

## Define reference helpers

Stimulus blocks refer to other blocks using `BlockReference` objects.

In [4]:
def dist_ref(name: str) -> obi.AllDistributionsReference:
    return obi.AllDistributionsReference(
        block_dict_name="distributions",
        block_name=name,
    )


def neuron_ref(name: str) -> obi.NeuronSetReference:
    return obi.NeuronSetReference(
        block_dict_name="neuron_sets",
        block_name=name,
    )

## Define distribution-based spike stimulus blocks

These examples show three common spike-timing patterns:

- regular spikes from a constant interval distribution
- Poisson-like spikes from an exponential interval distribution
- gamma-regular spikes from a gamma interval distribution

In [5]:
stimuli = {
    "regular_spike_input": obi.DistributionSpikeStimulus(
        source_neuron_set=neuron_ref("all"),
        targeted_neuron_set=neuron_ref("all"),
        duration=1000.0,
        distribution=dist_ref("regular_dist"),
        resample_each_repetition=False,
    ),
    "poisson_spike_input": obi.DistributionSpikeStimulus(
        source_neuron_set=neuron_ref("all"),
        targeted_neuron_set=neuron_ref("all"),
        duration=1000.0,
        distribution=dist_ref("poisson_dist"),
        resample_each_repetition=True,
    ),
    "gamma_spike_input": obi.DistributionSpikeStimulus(
        source_neuron_set=neuron_ref("all"),
        targeted_neuron_set=neuron_ref("all"),
        duration=1000.0,
        distribution=dist_ref("gamma_dist"),
        resample_each_repetition=True,
    ),
}

stimuli

{'regular_spike_input': DistributionSpikeStimulus(type='DistributionSpikeStimulus', timestamps=None, timestamp_offset=0.0, source_neuron_set=NeuronSetReference(block_dict_name='neuron_sets', block_name='all', type='NeuronSetReference'), targeted_neuron_set=NeuronSetReference(block_dict_name='neuron_sets', block_name='all', type='NeuronSetReference'), duration=1000.0, distribution=AllDistributionsReference(block_dict_name='distributions', block_name='regular_dist', type='AllDistributionsReference'), resample_each_repetition=False),
 'poisson_spike_input': DistributionSpikeStimulus(type='DistributionSpikeStimulus', timestamps=None, timestamp_offset=0.0, source_neuron_set=NeuronSetReference(block_dict_name='neuron_sets', block_name='all', type='NeuronSetReference'), targeted_neuron_set=NeuronSetReference(block_dict_name='neuron_sets', block_name='all', type='NeuronSetReference'), duration=1000.0, distribution=AllDistributionsReference(block_dict_name='distributions', block_name='poisson_d

## Summary

This notebook focuses on block composition only.

The blocks defined here can be added to a simulation form/config, where the
references will be resolved and the spike replay files will be generated during execution.

For a full runnable workflow, see the circuit/synaptome simulation examples.